<img src='../OUTILS/bandeau_MF.png' align='right' width='100%'/>

# <div style='background-color: #27ae60; color: white; padding: 20px; border-radius: 10px; text-align: center;'>🌍🛰️Manipulation de données satellitaires - Format NetCDF</div>

### 🎯Analyser un fichier NETCDF
### 🎯Produire une image
### 🎯Modifier la dynamique
### 🎯Extraire une valeur

## Workflow : lignes de commandes bash : GDAL & ImageMagick 

Ce TP utilise en grande partie les logiciels de la librairie **"gdal"** qui signifie : *geospatial data abstraction library*. Cette librairie est extrêmement utile pour manipuler les données issues des satellites météorologiques.

Pour ce TP nous allons utiliser des fichiers au format **NetCDF** issus de la production opérationnelle de Météo-France.  </br>


<div class="alert alert-info" role="alert">
<h3> ⚙️ Initialisation de l'environnement</h3>
Tout d'abord, il faut procéder à l'importation des librairies nécessaires à ce TP.
</div>

In [88]:
from datetime import datetime
import sys
import os
from osgeo import gdal
from PIL import Image
import subprocess
from IPython.display import display,HTML
os.environ['PATH'] = f"/opt/conda/env_MF_teledetection/bin:{os.environ['PATH']}" 
os.environ['PATH'] = f"~/.conda/envs/env_MF_teledetection/bin:{os.environ['PATH']}"
os.environ['GDAL_DATA'] = '/opt/conda/env_MF_teledetection/share/gdal'
os.environ['PROJ_LIB'] = '/opt/conda/env_MF_teledetection/share/proj'

Les fichiers NetCDF sont les suivants :</br>
 - Mmultic500mNC4_mtgi1_202604221200 </br>
 - Mmultic1kmNC4_mtgi1_202604221200 </br>
 - Mmultic2kmNC4_mtgi1_202604221200 </br>
Ils se trouvent dans ce répertoire : **`/stockage/DATA/NetCDF/20260422`**

Ces fichiers NetCDF multicanaux sont produits par Météo-France, au CMS, pour tous les satelittes géostationnaires.
Les différentes bandes y sont stockées

<p>Rappel des différents canaux</p>
</img><a href="../DOCS/Canaux.png" target="_blank">  <p> <img src='../DOCS/Canaux.png' alt='canaux' width='1000px'>  </p></a>

In [89]:
Mmultic500m="/stockage/DATA/NetCDF/20260422/Mmultic500mNC4_mtgi1_202604221200.nc"
Mmultic1km="/stockage/DATA/NetCDF/20260422/Mmultic1kmNC4_mtgi1_202604221200.nc"
Mmultic2km="/stockage/DATA/NetCDF/20260422/Mmultic2kmNC4_mtgi1_202604221200.nc"

<div class="alert alert-info alert-success">
<h3> 1 - 🔎 Visualiser le contenu du NetCDF </h3>
</div>

La commande gdalinfo permet d'afficher les informations qui se trouvent dans le fichier NetCDF. Usage : `gdalinfo mon_fichier`

Pour lancer les commandes gdal dans ce jupyter notebook il faut les précéder d'un point d'exclamation : `!gdalinfo`

In [ ]:
!gdalinfo $Mmultic2km

Driver utilisé : Driver: netCDF
→ Le fichier est au format NetCDF (format scientifique multidimensionnel)

Fichier source
Files: /stockage/.../Mmultic2kmNC4_mtgi1_202604221200.nc
→ Données satellite MTG (Meteosat Third Generation)

Taille raster principale : Size is 512, 512
→ Taille "globale" du dataset GDAL (souvent un conteneur)
→ ⚠️ Les vraies données sont dans les SUBDATASETS (voir plus bas)


#### METADATA (global attributes)


Zone couverte : Area_of_acquisition=globe
→ Couverture globale

Type de données : cdm_data_type=NetCDF
→ Convention standard Unidata

Dates
date_created=2026-04-22 12:05
time_coverage_start=12:00:07Z
time_coverage_end=12:09:34Z
→ Acquisition sur ~9 minutes (scan satellite)

Géolocalisation (approx)
lat: -80 → +80
lon: -80 → +80
→ Domaine visible depuis satellite géostationnaire

Institution
Meteo-France (CMS Lannion)

Description
summary=MTG01 Imager dataset
→ Données du capteur imageur MTG

### SUBDATASETS 

→ Le fichier contient plusieurs "bandes" scientifiques
→ Chaque SUBDATASET est une image 5568 x 5568 pixels

Les 16 canaux Meteosat-12 sont disponibles à 2 km :

Canaux visibles
VIS004, VIS005, VIS006, VIS008, VIS009 : Réflectance Top Of Atmosphere)

Canaux infrarouges proches 
IR_013, IR_016, IR_022 (réflectance ou température)

Température de brillance
IR_038, WV_063, WV_073, IR_087, IR_097, IR_105, IR_123, IR_133

Autre variable : dtime
→ Temps relatif par pixel



In [ ]:
!gdalinfo $Mmultic1km

10 canaux sont disponibles à la résolution 1km

In [ ]:
!gdalinfo $Mmultic500m

 2 canaux sont disponibles à la résolution 500m

In [ ]:
!gdalinfo NETCDF:"$Mmultic2km":VIS006

⚠️ WARNINGS </br>
Warning: dimension nx2km is not Longitude/X</br>
Warning: dimension ny2km is not Latitude/Y</br>
→ Satellite géostationnaire, les axes ne sont PAS en lat/lon, ce sont des coordonnées PROJETÉES (plan image satellite)</br>

Size is 5568 x 5568 </br>
→ Vraie taille de l’image</br>
→ Résolution : 2 km (voir plus bas)</br>
→ Donc disque complet ~ 11 000 km</br>

### PROJECTION (CRS)
PROJECTION: Geostationary Satellite (Sweep Y)</br>
→ Projection spécifique aux satellites type MTG / Meteosat</br>
→ Vue depuis l’espace (pas une projection terrestre classique)</br>
Paramètres clés :</br>
Longitude of origin = 0°</br>
→ Satellite centré sur Greenwich</br>
Satellite height = 35786400 m</br>
→ Orbite géostationnaire (~35 786 km)</br>

AXIS = (E, N) en mètres</br>
→ Coordonnées cartésiennes projetées (pas lat/lon)</br>

### GÉORÉFÉRENCEMENT
Origin = (-5568000, 5568000)</br>
Pixel Size = (2000, -2000)</br>
→ Pixel = 2 km</br>
→ Image centrée sur (0,0) = point sub-satellite</br>
Interprétation :</br>
  X : gauche → droite (-5.5 Mm → +5.5 Mm)</br>
  Y : haut → bas (+5.5 Mm → -5.5 Mm)</br>
→ C’est une grille "vue satellite"</br>

### EMPRISE
Coin haut gauche : (-5568 km, +5568 km)</br>
Coin bas droit : (+5568 km, -5568 km)</br>
→ Disque complet observable depuis le satellite</br>
Center = (0,0) → (0°E, 0°N)</br>
→ Point sous-satellite (Golfe de Guinée)</br>

###  BANDE VIS006
Longueur d’onde :</br>
0.640 µm (visible rouge)</br>
Type : toa_bidirectional_reflectance</br>

### CALIBRATION
scale_factor = 0.01</br>
add_offset = 0</br>
→ Conversion : reflectance = DN * 0.01 </br>

Exemple : </br>
DN = 10000 → réflectance = 100.0</br>
⚠️ Donc ici :
plage utile ≈ 0 → 100 (physique), mais stockée en int16</br>

###  NODATA
_FillValue = -32768</br>
→ Pixels hors disque / invalides</br>

### À RETENIR
✔️ Projection géostationnaire (pas lat/lon direct)</br>
✔️ Résolution 2 km</br>
✔️ Image centrée sur 0°E</br>
✔️ Données = Réflectance</br>
✔️ Conversion simple : DN * 0.01</br>
✔️ NoData = -32768</br>
✔️ Coordonnées en mètres (projection satellite)</br>



In [ ]:
!gdalinfo -mm NETCDF:"$Mmultic2km":VIS006

L'option -mm permet d'obtenir les valeur Min et Max du SUBDATASET, ou laors directement avec un grep

In [90]:
!gdalinfo -mm NETCDF:"$Mmultic2km":VIS006 |grep "Computed Min/Max" 

sh: line 1: getfattr: command not found
Warning 1: dimension #1 (nx2km) is not a Longitude/X dimension.
Warning 1: dimension #0 (ny2km) is not a Latitude/Y dimension.
    Computed Min/Max=264.000,12558.000


Les valeurs de réflectance vont ici de 264 à 12558 </br>
Rmq : Les réflectivités max sont souvent supérieures à 10000, donc > à 100 %. Cela peut-être le cas pour des nuages épais glacés, avec soleil rasant. 

La commande ncdump est aussi utile pour visualiser les métadonnées

In [ ]:
!ncdump -h /stockage/DATA/NetCDF/20260422/Mmultic2kmNC4_mtgi1_202604221200.nc | grep -A5 -B5 "VIS006" 

<div class="alert alert-info alert-success">
<h3> 2 - 🔎 Produire une image TIF à partir du canal VIS006 du NetCDF  </h3>
</div>

Production d'un image à partir du Multics

In [ ]:
cd ~/MF_DATA_MANIPULATION

In [ ]:
!mkdir -p RESULTS
output = 'RESULTS'
nom_fic = 'VIS006_mtg_20260422_1200'


Définition de la taille de l'image de sortie

In [ ]:
redim = '500x500'

Avec GDAL, nous allons transformer les valeurs physiques du NetCDF en nuances de gris, sur une plage 8-bits (byte), c'est-à-dire 256 valeurs. </br>
Il s'agit d'un étirement de contraste linéaire (contrast stretching) ou encore mise à l'échelle linéaire (linear scaling), permis par l'option : </br>
-scale entrée_min entrée_max sortie_min sortie_max


In [ ]:
!gdal_translate -scale 264 12558 0 255 -ot byte NETCDF:"$Mmultic2km":VIS006 $output/VIS006_mtg_20260422_1200_FD.jpg #>/dev/null 2>&1

In [ ]:
!convert -resize {redim} $output/VIS006_mtg_20260422_1200_FD.jpg $output/VIS006_mtg_20260422_1200_FD_{redim}.jpg
im1 = Image.open(f"{output}/VIS006_mtg_20260422_1200_FD_{redim}.jpg", 'r')
display(im1)

Nous allons ajuster la dynamique en changeant les bornes d'entrées (fenêtrage ou windowing), afin de rendre les valeurs plus blanches, donc plus visibles.</br>

In [ ]:
valeur_min_vis=264
valeur_max_vis=8000

In [ ]:
!gdal_translate -scale {valeur_min_vis} {valeur_max_vis} 0 255 -ot byte NETCDF:"$Mmultic2km":VIS006 $output/VIS006_mtg_20260422_1200_FD_dyn_modifiee.jpg >/dev/null 2>&1
!convert -resize {redim} $output/VIS006_mtg_20260422_1200_FD_dyn_modifiee.jpg $output/VIS006_mtg_20260422_1200_FD_dyn_modifiee_{redim}.jpg >/dev/null 2>&1
im2 = Image.open(f"{output}/VIS006_mtg_20260422_1200_FD_dyn_modifiee_{redim}.jpg", 'r')
display(im2)

Mais le gros problème est que toutes les valeurs de réflectance supérieures à 8000 (80%) seront à 255, soit un blanc brillant. </br>
Le blanc d'une grande partie des CB ou nuages élevés épais sera ainsi complètement saturé. Le signal sera perdu.

Une correction fréquemment utilisée en imagerie satellitaire pour récupérer de l'information ou éclaircir une image est le <b>gamma</b></br>
La correction Gamma est un ajustement qui permet de modifier la luminosité d’une image de façon non linéaire.</br>
Elle permet de modifier le rendu en fonction du besoin d'un algorithme ou de celui de la visualisation.</br>
C’est une courbe qui rend les zones sombres plus visibles sans trop éclaircir les zones claires. On parle aussi de « facteur de contraste ».</br>
Pour éclaircir une image, il faut augmenter le gamma

L'option -exponent est ajoutée à la commande GDAL.

In [ ]:
Gamma=2
ValGamma=1/Gamma #(GDAL prend en argument 1/gamma)

In [ ]:
!gdal_translate -scale {valeur_min_vis} {valeur_max_vis} 0 255 -exponent {ValGamma} -ot byte NETCDF:"$Mmultic2km":VIS006 $output/VIS006_mtg_20260422_1200_FD_dyn_gamma_modifiee.jpg >/dev/null 2>&1
!convert -resize {redim} $output/VIS006_mtg_20260422_1200_FD_dyn_gamma_modifiee.jpg $output/VIS006_mtg_20260422_1200_FD_dyn_gamma{Gamma}_{redim}.jpg >/dev/null 2>&1
im3 = Image.open(f"{output}/VIS006_mtg_20260422_1200_FD_dyn_gamma{Gamma}_{redim}.jpg", 'r')
display(im3)

A titre de comparaison, voici le produit opérationnelle pour cette même date

In [ ]:
!convert -resize {redim} /stockage/DATA/TIF/20260422/globeMvis006GTP.mtg.20260422.1200.24.tif.LT[3] $output/VIS006_mtg_20260422_1200_oper.jpg #>/dev/null 2>&1
im4 = Image.open(f"{output}/VIS006_mtg_20260422_1200_oper.jpg", 'r')
display(im4)

A noter que la production opérationnelle bénéficie de diverses corrections : diffusion, angle solaire et parallaxe.</br>
Ces corrections sont mieux visibles en zoomant:

In [ ]:
!gdalwarp -overwrite -t_srs '+proj=ortho lat_0=46.5 lon_0=1' -te -1120000 -630000 1120000 630000 -ts 960 540 /stockage/DATA/TIF/20260422/globeMvis006GTP.mtg.20260422.1200.24.tif.LT $output/VIS006_mtg_20260422_1200_oper_france.jpg
!gdal_translate -scale 264 12558 0 255 -ot byte NETCDF:"$Mmultic1km":VIS006 $output/VIS006_mtg_20260422_1200_FD_500m.tif
!gdalwarp -overwrite -t_srs '+proj=ortho lat_0=46.5 lon_0=1' -te -1120000 -630000 1120000 630000 -ts 960 540 $output/VIS006_mtg_20260422_1200_FD_500m.tif $output/VIS006_mtg_20260422_1200_FD_500m_france.jpg

In [ ]:
im5 = Image.open(f"{output}/VIS006_mtg_20260422_1200_oper_france.jpg", 'r')
display(im5)
im6 = Image.open(f"{output}/VIS006_mtg_20260422_1200_FD_500m_france.jpg", 'r')
display(im6)

💡 Pour comparer les images, les ouvrir dans des nouveaux onglets, ou utliser cette page d'aide à la comparaison de la <a href="http://e-sat.cms.meteo.fr/GALERIE/slider4_vierge.html" target="_blank"> GALERIE CMS</a> (Télécharger les 2 images et glisser/déposer)

<div class="alert alert-info alert-success">
<h3> 3 - 🔎 Produire une image TIF à partir du canal IR105 du NetCDF  </h3>
</div>

Les températures de brillances des données IR sont stockées en 16 bits dans les Multics, en centaines de degrés Celcius : la valeur numérique -50°C sera stockée en -5000

In [ ]:
nom_fic_IR = 'IR105_mtg_20260422_1200'

In [ ]:
!gdalinfo $Mmultic2km | grep SUBDATASET |grep IR

In [ ]:
!gdalinfo -mm NETCDF:"$Mmultic2km":IR_105

In [ ]:
valeur_min_ir=-9707 # valeur origine -9707
valeur_max_ir=5678 # valeur origine 5678

In [ ]:
!gdal_translate -scale {valeur_min_ir} {valeur_max_ir} 255 0 -ot byte NETCDF:"$Mmultic2km":IR_105 $output/IR_105_mtg_20260422_1200_FD_dyn_modifiee.jpg >/dev/null 2>&1
!convert -resize {redim} $output/IR_105_mtg_20260422_1200_FD_dyn_modifiee.jpg $output/IR_105_mtg_20260422_1200_FD_dyn_modifiee_{valeur_min_ir}_{valeur_max_ir}_{redim}.jpg >/dev/null 2>&1
im7 = Image.open(f"{output}/IR_105_mtg_20260422_1200_FD_dyn_modifiee_{valeur_min_ir}_{valeur_max_ir}_{redim}.jpg", 'r')
display(im7)

En Europe, la convention est de représenter les nuages froids en blanc. La dynamique de sortie est donc  : <b> -scale {valeur_min_ir} {valeur_max_ir} 255 0</b>. </br>
Mais ailleurs, comme aux USA, ce sont les nuages chauds qui sont représentés en blanc. La dynamique sera : <b> -scale {valeur_min_ir} {valeur_max_ir} 0 255</b>. </br>
Les seuils min et max pourront être adaptés dans tous les cas. </br>
👉 Relancer la production de l'image IR_105 en inversant la dynamique et/ou en modifiant les seuils.

En production opérationnelle, l'IR_105 est produite en 16 bits afin de bénéficier d'une plus grande plage de valeur</br>
-scale -10000 6500 330 0 -ot INT16 </br>
Exemple d'image IR opérationnelle en version jpg

In [ ]:
!gdalinfo -mm /stockage/DATA/TIF/20260422/globeMir105GTP.mtg.20260422.1200.24.tif.LT

La version oper :

In [ ]:
!gdal_translate -scale 0 328 0 255 -outsize 700 700 /stockage/DATA/TIF/20260422/globeMir105GTP.mtg.20260422.1200.24.tif.LT -ot Byte $output/IR105_mtg_20260422_1200_oper_dyn.jpg >/dev/null 2>&1
im8 = Image.open(f"{output}/IR105_mtg_20260422_1200_oper_dyn.jpg", 'r')
display(im8)

<div class="alert alert-info alert-success">
<h3> 4 - 🔎 Extraction de point d'un NetCDF  </h3>
</div>

GDAL permet d'extraire d'un NetCDF la valeur numérique  pour un pixel.</br>
🚨 A noter qu'il est préférable de récupérer cette information dans un fichier NetCDF "Multic" (Information en 16 bits) plutôt que dans un TIF.

In [ ]:
!gdallocationinfo -wgs84 NETCDF:"$Mmultic500m":VIS006  -3.47 48.75 2>/dev/null 

💡 Les valeurs stockées dans les NetCDF (Multics ici) sont presque toujours des <b>valeurs brutes</b> auxquelles il faut appliquer une formule.
   
Le but -> gagner de la place et donc du volume </br>
Un NetCDF Multic, codé en 16 bits, pèse environ 500 Mo. En 32 bits (Float), ce serait le double soit 1 Go. En 8 bits, ce serait seulement 250 Mo, mais on perdrait de l'information. </br>
Stockage 16 bits -> valeurs possibles : entier parmi 65 535 valeurs</br>

Pour les canaux visibles, il est indiqué dans le NetCDF:</br>
standard_name=toa_bidirectional_reflectance</br> 
scale_factor=0.01</br> 
add_offset=0</br> 

La valeur brute est ici de 6218. </br> 
Pour retrouver sa correspondance, il faut appliquer la formule </br> 
valeur = valeur brute * scale_factor + add_offset</br> 
valeur = 6218 * 0,01 + 0 = 62,18.</br> 
La reflectance est ici de 62,18 %

Cette valeur est donnée par GDAL : "Descaled Value"
    

Pour extraire la valeur de Temperature de brillance du canal IR10.5</br>

add_offset=273.15
scale_factor=0.01
standard_name=toa_brightness_temperature
units=K

In [ ]:
!gdallocationinfo -wgs84 NETCDF:"$Mmultic1km":IR_105  -3.47 48.75  #2>/dev/null 

Pour récupérer l'information en Kelvin, il faut ajouter le add_offset, et multiplier par le facteur "Scale"
Exemple : une valeur numérique de -3655 dans le multic correspond à : </br>
TB= 273,15 -3655 * 0.01 = 236,6 K.</br>
La valeur en °C est -36,55

Rmq: il est également possible de retrouver la valeur de la radiance spectrale à partir des métadonnées du NetCDF (Offset, Slope, nuc, alpha, beta)

La valeur renvoyée est par défaut est celle du point de grille le plus proche</br>
Pour obtenir la valeur interpolée : option -r 

In [ ]:
#-r cubic : très bon compromis rendu performance
!gdallocationinfo -wgs84 -r cubic NETCDF:"$Mmultic1km":IR_105 -3.47 48.75 2>/dev/null 

In [ ]:
Récupérer l'heure associée à cette valeur

In [ ]:
!gdallocationinfo -wgs84 -r cubic NETCDF:"$Mmultic1km":dtime  -3.47 48.75 2>/dev/null 

In [ ]:
!gdalinfo NETCDF:"$Mmultic1km":dtime

Le scan a été réalisé à 540 secondes (9 minutes) par rapport à l'heure du slot.
L'heure du slot est 12 h 00U TC.
L'heure du scan du point de coordonnées -3.47 48.75 est 12 h 09 UTC.

On retrouve cette information sur la vue générale suivante : </br>
</img><a href="../DOCS/chunks_latitude_et_heure_v20260505.jpg" target="_blank">  <p> <img src='../DOCS/chunks_latitude_et_heure_v20260505.jpg' alt='Scan chunks' width='800px'></p></a>